# Week 42, Day 3: Insertion and deletion

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w042d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def show(head):
    out = []
    while head is not None:
        out.append(head.value)
        head = head.next
    return out

head = Node(1, Node(2, Node(3, Node(4))))
before = head.next          # the node holding 2
before.next = before.next.next
print(show(head))
before.next = Node(99, before.next)
print(show(head))

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def from_list(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head

def show(head):
    out = []
    while head is not None:
        out.append(head.value)
        head = head.next
    return out

def insert_at(head, i, value):
    if i == 0:                              # no node before position 0: a special case
        return Node(value, head)
    before = head
    for _ in range(i - 1):
        if before is None:
            raise IndexError("position out of range")
        before = before.next
    if before is None:
        raise IndexError("position out of range")
    before.next = Node(value, before.next)
    return head

head = from_list(["a", "b", "d"])
head = insert_at(head, 2, "c")
head = insert_at(head, 0, "start")
head = insert_at(head, 5, "end")
print(show(head))

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def from_list(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head

def show(head):
    out = []
    while head is not None:
        out.append(head.value)
        head = head.next
    return out

def delete_first(head, target):
    dummy = Node(None, head)
    before = dummy
    while before.next is not None:
        if before.next.value == target:
            before.next = before.next.next       # the same line, whether it's the first node or not
            break
        before = before.next
    return dummy.next

head = from_list([5, 1, 5, 2])
head = delete_first(head, 5)
print(show(head))
head = delete_first(head, 2)
print(show(head), show(delete_first(None, 3)))

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def from_list(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head

def show(head):
    out = []
    while head is not None:
        out.append(head.value)
        head = head.next
    return out

def delete_all(head, target):
    dummy = Node(None, head)
    before = dummy
    while before.next is not None:
        if before.next.value == target:
            before.next = before.next.next       # stay put: check the new next
        else:
            before = before.next                  # move on only past a node we keep
    return dummy.next

print(show(delete_all(from_list([7, 7, 1, 7, 7, 2, 7]), 7)), show(delete_all(from_list([7, 7]), 7)))

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def from_list(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head

def show(head):
    out = []
    while head is not None:
        out.append(head.value)
        head = head.next
    return out

def merge(a, b):
    dummy = tail = Node(None)
    while a is not None and b is not None:
        if a.value <= b.value:                   # <= keeps the merge stable
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a if a is not None else b        # one list may have nodes left: attach them all
    return dummy.next

a, b = from_list([1, 4, 6]), from_list([2, 3, 7, 9])
first_of_a = a
merged = merge(a, b)
print(show(merged), merged is first_of_a)

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def from_list(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head

def show(head):
    out = []
    while head is not None:
        out.append(head.value)
        head = head.next
    return out

def partition(head, x):
    small = small_tail = Node(None)
    big = big_tail = Node(None)
    node = head
    while node is not None:
        if node.value < x:
            small_tail.next = small_tail = node
        else:
            big_tail.next = big_tail = node
        node = node.next
    big_tail.next = None                         # the old last link might point back into the list
    small_tail.next = big.next
    return small.next

print(show(partition(from_list([1, 4, 3, 2, 5, 2]), 3)))

> **Key idea.** In a singly linked list you can only change what comes after a node you hold, so walk with a pointer to the node **before** the change. Functions on bare heads return the (possibly new) head. A dummy head node gives the first node a node before it and removes the special cases; the real head is `dummy.next`. When deleting every match, move `before` on only past a node you keep. Merging, partitioning and splicing relink existing nodes: build results with a dummy and a tail pointer, attach leftovers, and end every list you make with `None`.

### Exercise: Insert at a position

Write `insert_at(head, i, value)`: insert a new node at position `i` (0 means at the front; `len` means at the end) and return the head. A position below 0 or above the length raises `IndexError`. Use a dummy head, so there's no special case for 0.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def insert_at(head, i, value):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef show(head):\n    out = []\n    while head is not None:\n        out.append(head.value)\n        head = head.next\n    return out\ndef test_insert():\n    \"inserting\"\n    h = build([\"b\", \"d\"])\n    h = insert_at(h, 1, \"c\")\n    h = insert_at(h, 0, \"a\")\n    h = insert_at(h, 4, \"e\")\n    assert show(h) == [\"a\", \"b\", \"c\", \"d\", \"e\"] and show(insert_at(None, 0, \"x\")) == [\"x\"]\n    for bad in [-1, 6]:\n        try:\n            insert_at(h, bad, \"z\")\n            assert False, bad\n        except IndexError:\n            pass\n    assert show(h) == [\"a\", \"b\", \"c\", \"d\", \"e\"], \"a refused insert changes nothing\"")

### Exercise: Delete the first match

Write `delete_first(head, target)`: remove the first node holding `target` (if there is one) and return the head. Use a dummy head.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def delete_first(head, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef show(head):\n    out = []\n    while head is not None:\n        out.append(head.value)\n        head = head.next\n    return out\ndef test_delete():\n    \"deleting\"\n    assert show(delete_first(build([3, 1, 3]), 3)) == [1, 3] and show(delete_first(build([1, 2, 3]), 3)) == [1, 2]\n    assert show(delete_first(build([1, 2]), 9)) == [1, 2] and delete_first(None, 1) is None and delete_first(build([4]), 4) is None")

### Exercise: Keeping it sorted

Write `sorted_insert(head, value)` for a list sorted in ascending order: insert a new node so it stays sorted (after any equal values), and return the head. Then `insertion_sort(head)`, which builds a **new** sorted list by inserting each value of the old one in turn.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def sorted_insert(head, value):
    ___

def insertion_sort(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef show(head):\n    out = []\n    while head is not None:\n        out.append(head.value)\n        head = head.next\n    return out\ndef test_sorted():\n    \"sorted insert\"\n    h = None\n    for v in [5, 1, 4, 1, 9]:\n        h = sorted_insert(h, v)\n    assert show(h) == [1, 1, 4, 5, 9]\n    rng = random.Random(2)\n    for _ in range(50):\n        vals = [rng.randint(0, 9) for _ in range(rng.randint(0, 12))]\n        assert show(insertion_sort(build(vals))) == sorted(vals)")

### Exercise: Merge by relinking

Write `merge(a, b)`: merge two sorted lists into one sorted list by **relinking their nodes**, without creating any node except a dummy, and return its head. On equal values, `a`'s node comes first. The checks make sure the result's nodes are the original node objects.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def merge(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef nodes(head):\n    out = []\n    while head is not None:\n        out.append(head)\n        head = head.next\n    return out\ndef test_merge():\n    \"merging\"\n    rng = random.Random(3)\n    for _ in range(100):\n        xs = sorted(rng.randint(0, 9) for _ in range(rng.randint(0, 8)))\n        ys = sorted(rng.randint(0, 9) for _ in range(rng.randint(0, 8)))\n        a, b = build([(x, \"a\") for x in xs]), build([(y, \"b\") for y in ys])\n        originals = {id(n) for n in nodes(a) + nodes(b)}\n        out = nodes(merge(a, b))\n        assert [n.value for n in out] == sorted([(x, \"a\") for x in xs] + [(y, \"b\") for y in ys])\n        assert {id(n) for n in out} == originals and len(out) == len(originals), \"relink the original nodes\"")

### Exercise: Odd positions first

Write `odd_even(head)`: rearrange the list so the nodes in positions 1, 3, 5, ... (counting from 1) come first, followed by those in positions 2, 4, 6, ..., keeping their order within each group, by relinking the nodes. Return the new head. Make sure the last node's `next` is `None`.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def odd_even(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef show(head, limit=100):\n    out = []\n    while head is not None and len(out) < limit:\n        out.append(head.value)\n        head = head.next\n    return out\ndef test_odd_even():\n    \"rearranging\"\n    for vals, want in [([1, 2, 3, 4, 5], [1, 3, 5, 2, 4]), ([1, 2, 3, 4], [1, 3, 2, 4]), ([], []), ([7], [7]), ([\"a\", \"b\"], [\"a\", \"b\"])]:\n        h = build(vals)\n        ids = set()\n        n = h\n        while n is not None:\n            ids.add(id(n)); n = n.next\n        out = odd_even(h)\n        assert show(out) == want, (vals, show(out))\n        n, seen = out, set()\n        while n is not None:\n            seen.add(id(n)); n = n.next\n        assert seen == ids, \"relink the same nodes\"")

### Exercise: Cutting out a run

Write `cut(head, i, j)`: remove the nodes at positions `i` to `j - 1` (half-open, like a slice; positions from 0) and return `(new_head, removed_head)`, where the removed nodes form their own list ending in `None`. If `i == j`, nothing is removed and `removed_head` is `None`. Positions must satisfy `0 <= i <= j <= length`, else `IndexError`.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def cut(head, i, j):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef show(head):\n    out = []\n    while head is not None:\n        out.append(head.value)\n        head = head.next\n    return out\ndef test_cut():\n    \"cutting\"\n    for vals, i, j in [(list(\"abcdef\"), 2, 4), (list(\"abcdef\"), 0, 2), (list(\"abcdef\"), 4, 6), (list(\"abc\"), 1, 1), (list(\"abc\"), 0, 3)]:\n        h, r = cut(build(vals), i, j)\n        assert show(h) == vals[:i] + vals[j:] and show(r) == vals[i:j], (vals, i, j)\n    for args in [(0, 4), (2, 1), (-1, 1)]:\n        try:\n            cut(build(\"abc\"), *args)\n            assert False, args\n        except IndexError:\n            pass")

### Bug hunt: The head that won't go

Without a dummy, this `delete_first` only ever looks at `node.next`, so it can't delete the head itself: deleting `"a"` from `a -> b -> c` leaves it unchanged, and this prints `['a', 'b', 'c']`. Fix it so it prints `['b', 'c']`.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def delete_first(head, target):
    node = head
    while node is not None and node.next is not None:
        if node.next.value == target:
            node.next = node.next.next
            break
        node = node.next
    return head

head = delete_first(Node("a", Node("b", Node("c"))), "a")
out = []
while head is not None:
    out.append(head.value)
    head = head.next
print(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the head too\"\n    expect_output(\"['b', 'c']\")")

### Bug hunt: Two in a row

After deleting a match, this moves `before` on anyway, so the next node is never checked, and two matches in a row leave one behind: this prints `[1, 0, 2]`. Fix it so it prints `[1, 2]`.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def delete_all(head, target):
    dummy = Node(None, head)
    before = dummy
    while before.next is not None:
        if before.next.value == target:
            before.next = before.next.next
        before = before.next
        if before is None:
            break
    return dummy.next

head = delete_all(Node(1, Node(0, Node(0, Node(2)))), 0)
out = []
while head is not None:
    out.append(head.value)
    head = head.next
print(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"check the new next\"\n    expect_output(\"[1, 2]\")")

### Bug hunt: The leftovers

`merge` stops when one list runs out but never attaches the rest of the other, so the biggest values disappear: this prints `[1, 2, 3, 4]`. Fix it so it prints `[1, 2, 3, 4, 8, 9]`.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def merge(a, b):
    dummy = tail = Node(None)
    while a is not None and b is not None:
        if a.value <= b.value:
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = None
    return dummy.next

head = merge(Node(1, Node(3)), Node(2, Node(4, Node(8, Node(9)))))
out = []
while head is not None:
    out.append(head.value)
    head = head.next
print(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"attach the rest\"\n    expect_output(\"[1, 2, 3, 4, 8, 9]\")")

### Puzzle: Deleting every match

Arrange the lines into a function that deletes every node holding `target`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

head = Node(1, Node(2, Node(3)))
dummy = Node(None, head)
dummy.next = dummy.next.next

p = ___   # dummy.next.value
q = ___   # head.value  (the old head object itself)
r = ___   # how many links change to splice a 3-node list in after a node, once you hold its tail
s = ___   # True or False: merge(a, b) by relinking needs O(n) extra memory
t = ___   # what a function that may delete the first node must do with the head: "return it" or "nothing"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q == 1 and r == 2, \"p, q and r\"\n    assert s is False and t == \"return it\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: two list surgeries

Write two functions that rearrange a list by relinking its nodes (no new nodes except dummies), each returning the new head:

1. `drop_repeated(head)`: for a **sorted** list, remove every value that appears **more than once**, keeping only values that appear exactly once: `1 2 3 3 4 4 5` becomes `1 2 5`.
2. `rotate_right(head, k)`: move the last `k` nodes to the front, in order: `1 2 3 4 5` rotated by 2 becomes `4 5 1 2 3`. `k` may be 0 or bigger than the length (rotating by the length changes nothing). Find the length once, and make it O(n).

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def drop_repeated(head):
    ___

def rotate_right(head, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef show(head, limit=100):\n    out = []\n    while head is not None and len(out) < limit:\n        out.append(head.value)\n        head = head.next\n    return out\ndef test_drop():\n    \"dropping repeats\"\n    assert show(drop_repeated(build([1, 2, 3, 3, 4, 4, 5]))) == [1, 2, 5] and show(drop_repeated(build([1, 1, 1, 2, 3]))) == [2, 3]\n    assert drop_repeated(build([2, 2])) is None and drop_repeated(None) is None\n    rng = random.Random(4)\n    for _ in range(100):\n        vals = sorted(rng.randint(0, 5) for _ in range(rng.randint(0, 10)))\n        assert show(drop_repeated(build(vals))) == [v for v in vals if vals.count(v) == 1], vals\ndef test_rotate():\n    \"rotating\"\n    for vals, k in [([1, 2, 3, 4, 5], 2), ([1, 2, 3], 0), ([1, 2, 3], 3), ([1, 2, 3], 7), ([9], 4), ([], 3)]:\n        n = len(vals)\n        want = vals[n - k % n:] + vals[:n - k % n] if n else []\n        assert show(rotate_right(build(vals), k)) == want, (vals, k)\n    big = build(list(range(20_000)))\n    assert show(rotate_right(big, 5), 6) == [19995, 19996, 19997, 19998, 19999, 0]")

## Warm-up

Deleting 3 and inserting 99 each changed **one** link, and both were made from the node **before** the change. That's the rule for singly linked lists: you can only change what comes after a node you're holding.

## The node before

In a singly linked list, a node knows what comes after it, not what comes before. So to delete a node, or insert in front of one, you walk with a pointer to the node **before** it:

Because the head itself may change (inserting at 0, deleting the first node), functions on bare heads **return** the new head, and callers write `head = insert_at(head, ...)`. Forgetting to keep the returned head is a classic bug.

## A dummy head

The special case for position 0 exists because the first node has no node before it. So give it one. A **dummy head** is a throwaway node placed in front of the list at the start of a function: now every real node has a node before it, the loop needs no special cases, and the real head is `dummy.next` at the end.

This is the singly linked version of yesterday's sentinel, made fresh for one function call instead of kept for the list's whole life.

## Deleting without skipping

Deleting **every** node holding a value has one trap: after deleting `before.next`, the new `before.next` hasn't been checked yet, so you must **not** move `before` on. Move it only when you keep a node:

It's the reader-and-writer idea from Week 41, in linked form: `before` is the writer, the last node kept.

## Relinking: merge and partition

Linked lists can be rearranged **without making a single new node**: only the links change. Merging two sorted lists is the classic (it's the merge step of merge sort, Week 19, and it's how merge sort is usually done on linked lists). A dummy head and a `tail` pointer collect the result:

The same tools split a list. To **partition** around a value (everything smaller first, keeping the order within each part), build two lists with two dummies and join them:

That `big_tail.next = None` matters: the last "big" node might be one that, in the original list, pointed at a small node, and without it the result would loop for ever.

(`small_tail.next = small_tail = node` uses Python's chained assignment: the right-hand side, `node`, is assigned to the targets **from left to right**, so `small_tail.next` is set first, on the old tail, then `small_tail` moves. Write it as two lines if you find that clearer.)

## Splicing runs

Once you hold the node before a run and the last node of the run, cutting the run out or putting a whole other list in is O(1): one or two links. The walk to find those nodes is the only O(n) part.

```text
cut out nodes 2..3:    a -> b -> [c -> d] -> e        becomes   a -> b -> e
                            ^before   ^last                       b.next = last.next

splice x -> y in after b:   a -> b -> c               becomes   a -> b -> x -> y -> c
                                                                   y.next = b.next; b.next = x
```

This is why linked lists suit editors, playlists and undo buffers, where whole stretches move around: an array would have to shift everything after the change.

## Practice

## Quiz

## Daily challenge

## Before you go

For each function today, point to the line that would go wrong without the dummy head, and say what case it would get wrong. Tomorrow: reversing a linked list, iteratively and recursively, which is the most famous linked-list problem of all.